In [1]:
from importlib import reload
import torch
import numpy as np
import time
torch.set_default_dtype(torch.float64)
np.random.seed(2)
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(device)
import numpy as np
import sys # Add the module path. Windows and Linux use different path separators; the raw string prevents escape sequences such as \n from being interpreted.
sys.path.append(r"../main_code/2d")
import generate_data,mesh,adaptive_int,visual,matrix_assemble,inverse_solver,mea_2d,net_2d,source_eval,error_general_2d,main
import matplotlib.pyplot as plt
CUDA_LAUNCH_BLOCKING=1

def kidney_curve(x, y, h, k, a):
    """
    Compute the implicit kidney-curve value at the given point (x, y).
    
    Parameters:
    x, y: Input coordinate points.
    h, k: Coordinates of the kidney-curve center used for translation.
    a:    Size parameter of the kidney curve.
    """
    x_shifted = x - h
    y_shifted = y - k
    
    # Avoid issues when a = 0.
    if a == 0:
        a = 1e-9

    # Implicit kidney-curve equation: (x_sh^2 + y_sh^2 - 4a^2)^3 - 108a^4 * y_sh^2 = 0
    term1 = (x_shifted**2 + y_shifted**2 - 4 * a**2)**3
    term2 = 108 * a**4 * y_shifted**2
    return term1 - term2

def Gauss_source(X,x_left,x_right,y_below,y_upper,center,r):
    # 1. Primary Gaussian source (simulated tumor).
    x=X[:,0]
    y=X[:,1]
    src1 = 1.2 * np.exp(-((x-0.3)**2 + (y-0.6)**2)/0.008)
    return src1

def kidney_source(X,x_left,x_right,y_below,y_upper,center,r,h=0.6, k=0.25, s=0.05):
    """Kidney-shaped source function."""
    x = X[:, 0]
    y = X[:, 1]
    # Kidney-shaped source with translation and scaling.
    kidney_src = np.where(kidney_curve(x, y, h, k, s) <= 0, 1.0, 0.0)
    return kidney_src

def elegant_source(X,x_left,x_right,y_below,y_upper,center,r,h=0.6, k=0.25, s=0.05):
    """Improved complex source function."""
    gauss= Gauss_source(X,x_left,x_right,y_below,y_upper,center,r)
    # Kidney-shaped source with translation and scaling.
    kidney_src = kidney_source(X, x_left, x_right, y_below, y_upper, center, r, h, k, s)

    return kidney_src +gauss



cuda:1


In [ ]:
import copy
np.random.seed(2)
torch.manual_seed(2)
torch.cuda.manual_seed_all(2) # Fix the random seed
Nx,Ny=1,1
Ix,Iy=4,4  # Number of grid cells in the x direction
nx,ny=3,3  # Gauss points per cell in the x direction
kk=[1,5,9,13,17,21,25,29,33,37,41,45,49,53,57,61,65,69,73,77,81,85,89,93,97,101] # Grid-point indices
tau_x_min,tau_x_max,tau_y_min,tau_y_max=0,1,0,1
b_x_min,b_x_max,b_y_min,b_y_max=-0.5,1.5,-0.5,1.5 # On boundary Gamma

num_batches_gauss=1
b_n=20
points_b=mea_2d.p_b(b_x_min,b_x_max,b_y_min,b_y_max,b_n) # Collect data on boundary Gamma
x_left,x_right,y_below,y_upper=0.29,0.49,0.3,0.7
r_true=0.2
center_true=np.array([0.6,0.25])
number_gene=50
eps=0.05
batch_number_rec_mea,mea=2,1
cupy_device = device.index if device.type == "cuda" else 0
number=120
a=0.05
S_int_true,F_mea_b,F_mea_db_x1,F_mea_db_x2=generate_data.boundary_data_circle(kk,120,points_b,kidney_source,0,1,0,1,center_true,1,a,eps,mea,batch_number_rec_mea,device,cupy_device)
S_int_true_g,F_mea_b_g,F_mea_db_x1_g,F_mea_db_x2_g=generate_data.boundary_data_rec(kk,120,points_b,Gauss_source,0,1,0,1,center_true,1,eps,mea,batch_number_rec_mea,device,cupy_device)
F=inverse_solver.F(F_mea_b+F_mea_b_g,F_mea_db_x1+F_mea_db_x1_g,F_mea_db_x2+F_mea_db_x2_g)
cells= mesh.create_initial_grid(tau_x_min,tau_x_max,tau_y_min,tau_y_max,Ix,nx,points_b) # Create the initial grid
initial_cells=copy.deepcopy(cells) # Make a copy
R_m=20


In [3]:
model_noise=torch.load("./noise=5%/model_noise.pth")
model_sig=torch.load("./noise=5%/model_sig.pth")
model_tanh=torch.load("./noise=5%/model_tanh.pth")

In [4]:
M=model_noise["hidden_layer_2.0.bias"].shape[0]+model_sig["hidden_layer_2.0.bias"].shape[0]+model_tanh["hidden_layer_2.0.bias"].shape[0]

In [5]:
af="Tanh"   # Initial activation setting
R_m=20
cupy_device = device.index if device.type == "cuda" else 0
model_tanh_IARFM = net_2d.local_rep(in_features=2, out_features=1, hidden_layers=1, M=M, M_noise=[],b1_noise=[],b2_noise=[],x_max=1, x_min=0, y_max=1, y_min=0, r_min=0.18, r_max=0.22,b1_min=[],b1_max=[],b2_min=[],b2_max=[],width_min=[],width_max=[],height_min=[],height_max=[],peak_min=[],peak_max=[],v_min=[],v_max=[],K_min=[],K_max=[],R_m_for_init=R_m,af="Tanh",Shape="general",device=device).to(device)


In [ ]:
iter_int= 5
Ix,Iy=4,4  # Number of grid cells in the x direction
delta=10**(-3)
nx=3
lamb_regu=np.logspace(-2,-1,1)
Qx,Qy=150,150
max_level=4
current_maxiter=1
af="Tanh"
Shape=[]
refine_threshold_S=1/100
refine_threshold_grad=1/300
cells_store,refinement_stats_store,w_,point_number,g_store,g_S,S_l2,S_num_store=main.ada_int(iter_int,delta,cells,nx,model_tanh_IARFM,M,af,Shape,[],kk,F,tau_x_min,tau_x_max,tau_y_min,tau_y_max,x_left,x_right,y_below,y_upper,center_true,r_true,elegant_source,points_b,lamb_regu,Qx,Qy,device,cupy_device,refine_threshold_S,refine_threshold_grad,current_maxiter,max_level)
